# GO Enrichment Analysis — **degu background**

This notebook is identical to `go_term_analysis.ipynb` **except** that the enrichment is
computed against a custom background: the **full degu annotated gene list** instead of
Enrichr's default (whole human GO library) background.

- **Foreground (unchanged):** the de novo novel genes from `../annotation-explanation/novel_genes.tsv`
- **Background (new):** all `gene` features in the final merged GFF
  `hifiasm_041425_denovoEnhanced_peaks2utr_sorted.gff3`, names taken from `ID=gene-*` and
  upper-cased — the same gene universe used in the annotation pie-chart notebook.
- A custom background requires **local GMT files** (gseapy computes locally). `human_GO_bp_2025.gmt`
  already existed; `human_GO_mf_2025.gmt` / `human_GO_cc_2025.gmt` were generated from the
  Enrichr 2025 libraries via `gp.get_library`.

Outputs are saved with a `_degubg` suffix so no original files are touched.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt

In [ ]:
go_df=pd.read_csv(f"{PROJ_ROOT}/figure/annotation-explanation/novel_genes.tsv", sep="\t", index_col=0)


In [ ]:
# Example gene list (replace with your genes)
gene_list = go_df["gene_name"]
cleaned = gene_list.str.replace(r"-(?:DL|RL|L)\d+$", "", regex=True)
gene_list=cleaned.unique()
print(len(gene_list))
gene_list

In [ ]:
## ============================================================
## CHANGE FROM THE ORIGINAL: custom (degu) background.
## Foreground gene list is unchanged. Only the background is new.
## ============================================================

import re

# ---- Build the degu background: all annotated degu genes ----
# Same universe as the annotation pie-chart notebook: every "gene" feature
# in the final merged GFF, ID=gene-<name>, upper-cased, deduplicated.
final_gff = f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm_041425_denovoEnhanced_peaks2utr_sorted.gff3"
background_genes = set()
with open(final_gff) as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 9 or parts[2] != "gene":
            continue
        m = re.search(r"ID=gene-([^;]+)", parts[8])
        if m:
            background_genes.add(m.group(1).upper())
background_genes = sorted(background_genes)
print("degu background gene list size:", len(background_genes))

# ---- Foreground (unchanged: de novo novel genes) ----
gene_list = go_df["gene_name"]

# ---- GO enrichment against the degu background ----
go_results = gp.enrichr(
    gene_list=gene_list,
    gene_sets=['human_GO_bp_2025.gmt', 'human_GO_mf_2025.gmt', 'human_GO_cc_2025.gmt'],
    organism='human',                # symbol space of the GMTs
    background=background_genes,     # <-- THE degu background (change from original)
    outdir=None,                     # don't write to disk
)

# Convert to DataFrame and save
go_df = go_results.results
go_df.to_csv('go_enrichment_results_degubg.csv', index=False)

# Plot top terms
plt.figure(figsize=(10, 8))
gp.dotplot(go_results.results, 
           title='GO Enrichment Analysis (degu background)',
           cutoff=0.05,  # p-value cutoff
           top_term=10,  # show top 10 terms
           figsize=(10, 8),
           show_ring=True,
           cmap='viridis')
plt.savefig('go_enrichment_plot_degubg.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
go_df['GO_ID'] = go_df['Term'].str.extract(r'\((GO:\d+)\)')
go_df.head()

In [ ]:
go_df.to_csv("GO_terms_novel_genes_degubg.tsv", sep="\t")
